# 04 · Urban heat and greening scenarios

**Question:** How does a raster intervention affect population-weighted exposure?

**Runtime:** Python / Pyodide in JupyterLite. Choose **Run → Run All Cells**.
Each notebook is independent; start with a fresh kernel. Initial runtime/package downloads need internet.
The initial baseline uses **synthetic educational fixtures** near a Gaborone-area anchor.
Extended investigations explicitly identify bundled real OSM, Sentinel or terrain extracts where used;
these do not validate the synthetic population, mobility, exposure or scenario assumptions.

**Astra experiment:** Ask Astra to explain the map and distinguish area-average from population-weighted changes; score the numerical claims.
Run the numerical baseline first, then give Astra the exported evidence and the prompt below.
The baseline does not call or run GPT-6 Astra. No API key is needed.


In [ ]:
import sys, json, math, time
from pathlib import Path
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "matplotlib"])
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from twin import *
style()
print("Runtime:", sys.platform, "| data: synthetic | seed: 42")


## Synthetic raster, not satellite temperature
The heat field has degree-Celsius labels for teaching, but comes from an invented equation.
Nearest-pixel sampling is explicit; actual raster work needs CRS, nodata, acquisition time and resolution checks.


In [ ]:
c=city(); pop=allocate(c['capacity'],1200)
xx,yy,green,heat,elevation=raster()
cooling_c=2.5; radius_m=450.; center=(1400,700)
effect=cooling_c*np.exp(-((xx-center[0])**2+(yy-center[1])**2)/(2*radius_m**2))
after=heat-effect
ix=np.clip(np.rint(c['xy'][:,0]/2000*(heat.shape[1]-1)).astype(int),0,heat.shape[1]-1)
iy=np.clip(np.rint(c['xy'][:,1]/2000*(heat.shape[0]-1)).astype(int),0,heat.shape[0]-1)
before_mean=float(np.average(heat[iy,ix],weights=pop)); after_mean=float(np.average(after[iy,ix],weights=pop))
assert np.isfinite(after).all() and (after<=heat).all()
print('Population-weighted synthetic temperature:',round(before_mean,2),'->',round(after_mean,2))


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,4))
for ax,field,title in zip(axes,[heat,after,effect],['Baseline','Greening scenario','Cooling difference']):
    m=ax.imshow(field,origin='lower',extent=[0,2000,0,2000],cmap='YlOrRd',vmin=heat.min() if title!='Cooling difference' else 0,vmax=heat.max() if title!='Cooling difference' else cooling_c)
    map_axes(ax,title); fig.colorbar(m,ax=ax,label='Synthetic degrees C')
plt.tight_layout(); plt.show()
export_json('04_heat.json',dict(synthetic=True,baseline_c=before_mean,scenario_c=after_mean,cooling_c=cooling_c,radius_m=radius_m))


## Teaching lab: Explain raster evidence
**Before using AI:** write a prediction, run the numerical baseline, and explain one surprising result.

**Astra evidence:** send only 04_heat.json and the displayed maps. Use a fresh conversation and record the exact model,
date, interface, tools and prompt. A task with a supplied result tests interpretation, not independent calculation.

**Copyable Astra prompt:**
> Compute the population-weighted cooling from the exported baseline and scenario. Explain why this need not equal an area-average change.

**Implementation brief — review the runnable extension below:**
> Calculate both area-average and population-weighted cooling; expose the sampling method and units in the export. Edit the authoring source in `scripts/make_notebooks.py` (or
> `scripts/teaching_labs.py` for labs 11–13), regenerate the notebooks and preserve browser compatibility.
> Use NumPy, Matplotlib and the standard library. Include a small reference case that can be checked by hand.

**Acceptance evidence:** Zero cooling intensity produces zero change; weighting uses all 1200 residents; raster rows correspond to northing.

The extended investigation later in this notebook implements this line of work with maps and checks.
Use the brief to review the implementation or ask Codex for a further controlled variation.

**Share back:** save your original prediction, changed parameter, result, model response and one limitation.
Download the edited notebook and exports. If you have no model access, exchange explanations with a partner
and evaluate them with the same criteria; do not label that activity as an Astra trial.


## Extended investigation: Inspect real Sentinel reflectance, masks and scale

This bundled 20 m Sentinel-2 chip is real surface-reflectance evidence near the teaching anchor, acquired January 23, 2025. Red/NIR scale and offset were applied during preprocessing; SCL is categorical and resampled with nearest-neighbor logic. We retain vegetation, bare-ground and water classes (4, 5, 6). An additional artificial cloud patch demonstrates missing-data behavior and is explicitly labeled. NDVI is not land-surface or air temperature, and does not validate the earlier cooling equation.

**Read the concept map first:** identify the input, method, evidence and limit. Predict the spatial pattern before running the comparison.


In [ ]:
from extension_tools import concept_map, export_investigation
concept_map('Inspect real Sentinel reflectance, masks and scale', ['Sentinel red / NIR / SCL', 'Scale, mask, compute NDVI', 'Valid pixels + uncertainty', 'NDVI is not temperature'])
plt.show()


In [ ]:
from extension_tools import load_data
from matplotlib.colors import BoundaryNorm, ListedColormap
ex_chip=load_data('sentinel_chip.json'); ex_red=np.array(ex_chip['red']); ex_nir=np.array(ex_chip['nir']); ex_scl=np.array(ex_chip['scl'])
ex_valid=np.isin(ex_scl,[4,5,6]) & ((ex_red+ex_nir)>0)
ex_ndvi=np.full(ex_red.shape,np.nan); ex_ndvi[ex_valid]=(ex_nir[ex_valid]-ex_red[ex_valid])/(ex_nir[ex_valid]+ex_red[ex_valid])
assert np.isfinite(ex_ndvi[ex_valid]).all()
ex_cloud=ex_valid.copy(); ex_cloud[20:35,25:45]=False
ex_masked=np.where(ex_cloud,ex_ndvi,np.nan)
fig,axes=plt.subplots(1,3,figsize=(14,4)); bounds=ex_chip['bounds']; extent=[bounds[0],bounds[2],bounds[1],bounds[3]]
for ax,field,title,cmap in zip(axes,[ex_ndvi,ex_scl,ex_masked],['Real NDVI, original valid pixels','Observed SCL classes','Artificial cloud added for teaching'],['RdYlGn','tab10','RdYlGn']):
    options=dict(vmin=-1,vmax=1) if title!='Observed SCL classes' else dict(norm=BoundaryNorm([3.5,4.5,5.5,6.5],3))
    if title=='Observed SCL classes': cmap=ListedColormap(['#328a55','#c9a16a','#398ac4'])
    image=ax.imshow(field,extent=extent,origin='upper',cmap=cmap,**options); colorbar=fig.colorbar(image,ax=ax,shrink=.72,pad=.08)
    if title=='Observed SCL classes': colorbar.set_ticks([4,5,6],labels=['Vegetation','Not vegetated','Water'])
    short_title={'Real NDVI, original valid pixels':'Observed NDVI','Observed SCL classes':'Observed SCL classes','Artificial cloud added for teaching':'NDVI with artificial cloud'}[title]
    ax.set(title=short_title,xlabel='UTM easting (m)',ylabel='UTM northing (m)'); ax.ticklabel_format(style='plain',useOffset=False)
    ax.title.set_fontsize(10); ax.tick_params(labelsize=8); colorbar.ax.tick_params(labelsize=8)
    ax.set_xticks(np.linspace(extent[0],extent[1],3))
fig.text(.01,.01,'Contains modified Copernicus Sentinel data (2025) · ESA / Element 84 · '+ex_chip['crs'],fontsize=9)
fig.tight_layout(rect=[0,.05,1,1]); extension_figure=fig
print('Observed mean NDVI:',float(np.nanmean(ex_ndvi)),'| after artificial cloud mask:',float(np.nanmean(ex_masked)))
print('Original area vs population-weighted cooling:',float(effect.mean()),float(before_mean-after_mean),'synthetic °C')
assert pop.sum()==1200 and np.array_equal(heat-0*effect,heat)
extension_evidence=dict(provenance=ex_chip['provenance'],original_valid_pixels=int(ex_valid.sum()),retained_pixels=int(ex_cloud.sum()),pixel_m=20,artificial_cloud=True,synthetic_cooling=dict(area_average_c=float(effect.mean()),population_weighted_c=float(before_mean-after_mean),sampling='Nearest raster pixel; row = northing; all 1200 residents included'))
extension_expected=dict(hidden_area_m2=int((ex_valid&~ex_cloud).sum())*400,temperature_observed=False)
extension_task='Calculate area hidden by the added artificial cloud and determine whether red/NIR NDVI observes temperature. Return hidden_area_m2 and temperature_observed.'


In [ ]:
extension_prompt,extension_reference=export_investigation('04',extension_evidence,extension_expected,extension_task,extension_figure)
plt.show()
print('Send only 04_extension_prompt.json and optionally 04_extension_map.png to Astra. Keep the reference in this kernel.')


### Ask, check and explain the spatial result
Download `04_extension_prompt.json` and `04_extension_map.png`. Send the evidence in a fresh Astra
session with the exact task and output fields. For a separate image-only experiment, supply only the image
and task wording, record that condition, and allow clarification if the image does not contain enough information.
Never send the reference or this complete notebook for a blind trial.

Paste the unedited response below or upload a UTF-8 response file. The default is deliberately empty:
**no model trial has run**. After recording a real answer, set `extension_source='MODEL'` and complete the metadata.
The numeric tolerance is 0.01 in each requested field's declared units; labels and booleans require exact agreement.
Keep that rule fixed before examining a response. Inspect the explanation separately for unsupported claims.

**Visual reflection:** point to one map pattern, cite the numerical check that supports it, and name one thing
the visual cannot establish. Re-run one parameter change and preserve both maps. The implementation above
is available for inspection and modification; it is not a measured claim about model accuracy.


In [ ]:
from experiments import grade_answer
extension_source='NOT RUN'
extension_model=''; extension_date=''; extension_interface=''; extension_condition='text-and-map'
extension_response_file=''; extension_response_text=''
if extension_response_file:
    extension_response_text=Path(extension_response_file).read_text(encoding='utf-8')
if extension_source=='MODEL':
    assert extension_model and extension_date and extension_interface and extension_response_text, 'Record the actual model, date, interface and response'
extension_tolerances={key:.01 for key,value in extension_reference.items() if type(value) in (int,float)}
extension_grade=grade_answer(extension_response_text,extension_reference,extension_tolerances) if extension_response_text else dict(status='NOT RUN')
display(extension_grade)
export_json('04_extension_trial.json',dict(source=extension_source,model=extension_model,date=extension_date,
    interface=extension_interface,condition=extension_condition,prompt=extension_prompt,response_text=extension_response_text,
    tolerances=extension_tolerances,reference=extension_reference,grade=extension_grade))


## Save the investigation
Download the edited notebook, evidence, map and trial record from `exports/`. Browser storage does not
commit changes to GitHub. Keep data attribution and the distinction between observed data, synthetic
assumptions and actual model responses when sharing. The extended code and its checks above are part of
this notebook; use them as the starting point for your next controlled comparison.
